In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, from_json, current_timestamp, get_json_object, explode, first, explode_outer
from pyspark.sql.types import StructType, StructField, StringType, ArrayType, MapType, DoubleType, TimestampType, IntegerType
import pyspark.sql.functions as F

In [ ]:
try:
    spark.stop()
except:
    pass

In [ ]:
spark = (SparkSession.builder.appName("HealthcareDataProcessing_condition")
.config("spark.sql.files.ignoreCorruptFiles", "true")
.config("spark.driver.memory", "4g") 
.config("spark.executor.memory", "4g") 
.config("spark.memory.offHeap.enabled", "true") 
.config("spark.memory.offHeap.size", "2g") 
.master("local[*]")
.getOrCreate())

In [ ]:
bronze_path = "../../data_lake/bronze/batch_data/resource_type=Condition/"
silver_condition_path = "../../data_lake/silver/silver_condition/"

In [ ]:
general_coding_schema = StructType([
        StructField("system", StringType(), True),
        StructField("code", StringType(), True),
        StructField("display", StringType(), True)
    ])

type_schema = StructType([
        StructField("coding", ArrayType(general_coding_schema), True),
        StructField("text", StringType(), True)
    ])

In [ ]:
df_condition = spark.read.format("parquet").load(bronze_path)

In [ ]:
df_inter = df_condition.select(
    col("resource.id").alias("condition_id"),
    from_json(col("resource.clinicalStatus").cast("string"), type_schema).alias("clinicalStatus"),
    from_json(col("resource.verificationStatus").cast("string"), type_schema).alias("verification_status"),
    from_json(col("resource.category"), ArrayType(type_schema)).alias("category"),
    from_json(col("resource.code"), type_schema).alias("code"),
    F.split(F.get_json_object(col("resource.subject"), "$.reference"), ":")[2].alias("patient_id"),
    F.split(F.get_json_object(col("resource.encounter"), "$.reference"), ":")[2].alias("encounter_id"),
    col("resource.onsetDateTime").cast("timestamp").alias("onset_date_time"),
    col("resource.abatementDateTime").cast("timestamp").alias("abatement_date_time"),
    col("resource.recordedDate").cast("timestamp").alias("recorded_date"),
    col("input_file_name")
)

In [ ]:
df_inter_2 = df_inter.select(
    col("condition_id"),
    col("clinicalStatus.coding")[0]["code"].alias("clinical_status_coding"),
    col("verification_status.coding")[0]["code"].alias("verification_status_coding"),
    col("category")[0]["coding"][0]["code"].alias("category_coding"),
    col("code.coding")[0]["code"].alias("code_coding"),
    col("patient_id"),
    col("encounter_id"),
    col("onset_date_time"),
    col("abatement_date_time"),
    col("recorded_date"),
    col("input_file_name"),
    current_timestamp().alias("silver_timestamp")
)

In [ ]:
df_inter_2.write.mode("overwrite").format("parquet").save(silver_condition_path)

In [ ]:
spark.stop()